# Workflow Optimizer — V0

**What this does.** You describe a task (e.g. *"grade-school math word problems, integer answers"*). The notebook then has an AI agent **design candidate solution strategies and iteratively make them cheaper** without losing accuracy, runs them on real examples, and lets you pick the strategy with the accuracy/cost tradeoff you want.

**The key idea: a "workflow" is a small Python function.** Every candidate strategy is written as a function with this exact shape:

```python
def solve(question, llm):
    # ...any Python you want...
    return answer
```

- `question` — one input from the dataset (a string).
- `llm` — **a function the notebook hands to your program.** Calling `llm("some prompt")` sends that prompt to a model and returns the model's reply as a string. It is *not* a string or a model name; it's the single gateway to the model (defined in the *"one model call"* cell below). Everything a workflow does with a model goes through it — which is how the notebook meters cost and enforces a budget.
- `answer` — whatever the strategy produced; the notebook then extracts and grades it.

Because `solve` is ordinary code, it can express **any** strategy — a single `llm(...)` call, chain-of-thought, sampling 5 times and majority-voting, breaking the problem into steps, or routing easy questions to a cheap model and hard ones to an expensive one. Nothing is hard-coded; the agent writes the code.

**The pipeline (this notebook, top to bottom):**
1. **Profile the task** — a model reads your description and works out how answers should be *extracted* and *checked*, and generates example data if you didn't supply any.
2. **Optimize (a loop)** — a Claude Code agent designs candidate `solve` programs, then **iterates**: each round it sees the best workflows so far and tries to make *cheaper* ones that keep accuracy, building up an archive of candidates.
3. **Choose** — rank the surviving tradeoffs on a held-out test split, then pick the methodology you want (each is a real point on the accuracy/cost **Pareto frontier** — nothing else beats it on *both* price and accuracy at once).

**Setup.** Every code cell makes real Anthropic API calls, so set `ANTHROPIC_API_KEY` first. Each workflow runs under a per-query budget (a cap on model calls + tokens); a program that overspends or crashes simply scores 0.

## Setup

The model **prices** (USD per million tokens) and `cost_usd(model, usage)`, which turns a call's token counts into dollars. It's **cache-aware**: prompt-cache *writes* bill 1.25× the input rate and *reads* only 0.10× (≈90% off), so resending the same prompt to the same model is cheap. `client` is the Anthropic client every model call goes through.

In [1]:
import os, re, json, statistics

import anthropic
import pandas as pd
import matplotlib.pyplot as plt

# ---- Pricing: USD per 1,000,000 tokens (input, output) --------------------
PRICES = {
    "claude-haiku-4-5": (1.0,  5.0),
    "claude-sonnet-5":  (3.0, 15.0),
    "claude-opus-4-8":  (5.0, 25.0),
}

# Prompt-cache multipliers on the INPUT rate. Writing a prompt into the cache
# costs a bit more once (1.25x); reading it back later is ~90% off (0.10x).
CACHE_WRITE_MULT = 1.25
CACHE_READ_MULT = 0.10

def cost_usd(model, usage):
    # usage is a dict of token counts: {"input", "output", "cache_write", "cache_read"}.
    # Regular input + output bill at the base rate; cache writes/reads use the
    # multipliers above. The cache is keyed by model server-side, so resending the
    # same prompt to the SAME model bills the cheap read rate, while sending it to a
    # DIFFERENT model is a fresh write (a miss, no discount).
    price_in, price_out = PRICES[model]
    tokens = (usage["input"]       * price_in
              + usage["cache_write"] * price_in * CACHE_WRITE_MULT
              + usage["cache_read"]  * price_in * CACHE_READ_MULT
              + usage["output"]      * price_out)
    return tokens / 1_000_000

# ---- Anthropic API client -------------------------------------------------
# Every rollout is a real API call, so a key is required.
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your shell env before running.")
client = anthropic.Anthropic()

print("Anthropic client ready.")

Anthropic client ready.


## Define the task — *the only cell you edit per task*

`SEED_PROMPT` describes the task in plain English. `DATASET` is optional — your own labeled examples as a list of `{"question": ..., "answer": ...}`; leave it `None` and the notebook generates some. Everything below adapts to whatever you put here.

In [3]:
# ---- Define the problem (the ONLY per-task input) -------------------------
# Describe the task in SEED_PROMPT. Optionally supply your own labeled DATASET
# as a list of {"question": <input str>, "answer": <target>}; leave it None to
# have one generated. The answer format (numeric / exact-match / free-form) is
# figured out automatically in the profiling step below.
SEED_PROMPT = ("Build a system that generates clinical notes for a given doctor-patient conversation")
DATASET = None

## The models a workflow may use

A pool of models, listed cheapest → most expensive. A workflow chooses among them per call with `llm(..., model=...)`; the first one is the default when a call doesn't specify a model.

In [5]:
# ---- Models to search over ------------------------------------------------
# The optimizer searches over (model x strategy). The strategies are NOT written
# here — the model designs them further down. Add "claude-opus-4-8" to include Opus.
MODELS = ["claude-haiku-4-5", "claude-sonnet-5", "claude-opus-4-8"]
print("Models:", ", ".join(MODELS))

Models: claude-haiku-4-5, claude-sonnet-5, claude-opus-4-8


## The one model call — what `llm` actually is

This defines the single function that reaches a model. The `llm` your `solve(question, llm)` receives **is this function** (wrapped with cost metering — see *The runtime* cell later). You call it as `llm(prompt)` and get back the model's reply as a string.

A workflow can also pass:
- `model=` — route this call to a specific model.
- `system=` — a system prompt for this call.
- `tools=["code_execution"]` and/or `["web_search"]` — let the model run Python or search the web (these run server-side; results come back in the reply).
- `effort="low"…"max"` — let a large model think step-by-step before answering.

Routing *every* model call through this one place is what lets the notebook measure cost and enforce a per-query budget, no matter what the program does. (`_call_api` is the raw call; `Runtime.llm`, defined later, wraps it with metering and is what a program actually receives.)

**Prompt caching.** The prompt and system prompt carry a cache breakpoint, so resending the *same* prompt to the *same* model reads it from cache — the reused tokens bill at ~10% of the input rate instead of full price (the first send pays a one-time 25% write surcharge). The cache is keyed by model, so the *same* prompt to a *different* model is a fresh miss with no discount. Only prompts above the model's minimum (~1–2k tokens) cache at all; shorter ones bill normally. `_call_api` returns `usage` split into `input` / `output` / `cache_write` / `cache_read`, and `cost_usd` prices each accordingly.

In [8]:
# ---- One model call (the only way a workflow reaches a model) --------------
# A program calls llm(...) with any of these; they map straight onto the API here.
# Server-side tools (code execution, web search) run on Anthropic's side, so
# there's nothing to execute locally — the model uses them and the results come
# back in the same reply. `effort` turns on the model's own step-by-step thinking.
#
# The prompt (and system) carry a cache breakpoint, so resending the SAME prompt to
# the SAME model reads it from cache instead of paying full price. The cache is
# server-side and keyed by model, so a different model is always a fresh miss.

TOOL_DEFS = {   # name a program passes -> the tool definition sent to the API
    "code_execution": {"type": "code_execution_20260521", "name": "code_execution"},
    "web_search":     {"type": "web_search_20260209", "name": "web_search"},
}
THINKING_MODELS = {"claude-sonnet-5", "claude-opus-4-8"}   # models that support effort

def _call_api(model, prompt, max_tokens, system=None, tools=None, effort=None):
    # A cache breakpoint on the prompt -> identical resends to this model are cheap.
    content = [{"type": "text", "text": str(prompt),
                "cache_control": {"type": "ephemeral"}}] # cache_control ephemeral means auto-expiring cache (typically 5m)
    request = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": content}],
    }
    if system:
        request["system"] = [{"type": "text", "text": system,
                              "cache_control": {"type": "ephemeral"}}]
    if tools:
        request["tools"] = []
        for name in tools:
            request["tools"].append(TOOL_DEFS[name])
    if effort and model in THINKING_MODELS:
        request["thinking"] = {"type": "adaptive"}
        request["output_config"] = {"effort": effort}
        request["max_tokens"] = max_tokens   # remember to leave room for thinking
    else:
        request["thinking"] = {"type": "disabled"}        # default: the strategy is the only knob

    text = ""
    # Token counts, split so cost_usd can price cached vs. fresh tokens differently.
    usage = {"input": 0, "output": 0, "cache_write": 0, "cache_read": 0} 
    for _ in range(5):                     # a tool-using turn can pause; resume it
        message = client.messages.create(**request)
        usage["input"]  += message.usage.input_tokens # anthropic returns thes from the usage block in the response
        usage["output"] += message.usage.output_tokens
        usage["cache_write"] += getattr(message.usage, "cache_creation_input_tokens", 0) or 0
        usage["cache_read"]  += getattr(message.usage, "cache_read_input_tokens", 0) or 0
        for block in message.content:
            if block.type == "text":
                text += block.text
        if message.stop_reason != "pause_turn":
            break
        request["messages"].append({"role": "assistant", "content": message.content})
    return text, usage

**Examples — pricing + the one model call.** `_call_api` returns `(text, usage)`, where `usage` counts `input` / `output` / `cache_write` / `cache_read` tokens; `cost_usd(model, usage)` prices them. Cache reads bill ~10% of the input rate, so resending the same prompt to the same model is cheap.

```python
# haiku input is $1/M. The same 2,000-token prompt, three ways:
cost_usd("claude-haiku-4-5", {"input": 2000, "output": 0, "cache_write": 0, "cache_read": 0})  # uncached   -> 0.0020
cost_usd("claude-haiku-4-5", {"input": 0, "output": 0, "cache_write": 2000, "cache_read": 0})  # first send -> 0.0025
cost_usd("claude-haiku-4-5", {"input": 0, "output": 0, "cache_write": 0, "cache_read": 2000})  # sent again -> 0.0002

# _call_api returns the split usage (this short prompt is below the cache minimum, so no caching):
_call_api("claude-haiku-4-5", "What is 2+2?", 32)   # -> ("4", {"input": 14, "output": 3, "cache_write": 0, "cache_read": 0})
_call_api("claude-sonnet-5", "Solve it.", 512, system="You are a careful math tutor.")
_call_api("claude-sonnet-5", "Compute 47 * 53.", 512, tools=["code_execution"])
_call_api("claude-opus-4-8", "A tricky word problem...", 512, effort="high")
```

## Reading and grading an answer

A workflow returns free-form text, so we (1) pull the answer out and (2) decide if it's right — two jobs with different trust needs.

- **Extraction is task-specific**, so the profiler (next cell) *writes* an `extract(text)` for your task and validates it against the gold data. `extract_answer(text, extract_type)` here is the **deterministic fallback** used when that validation fails — `last_number`, `last_line`, or the `full` text.
- **Checking is the ground-truth comparator**, so it stays fixed and auditable: `check_answer(prediction, gold, check_type)` — `numeric`, `exact`, or `llm_judge` (ask a cheap model). The profiler picks `check_type`.

`extract_number` (the last number in a string) is a helper both the fallback and the LLM-written extractor may use.

In [ ]:
# ---- Pull the answer out of text, and check it against the gold answer -----
# The task profile (next cell) picks the types:
#   extract_type in {"last_number", "last_line", "full"}
#   check_type   in {"numeric", "exact", "llm_judge"}

def extract_number(text):
    # The last number appearing in the text, or None. Also handed to workflow
    # programs so they can parse their own intermediate results.
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text or "")
    if not numbers:
        return None
    try:
        return float(numbers[-1].replace(",", ""))
    except ValueError:
        return None

def extract_answer(text, extract_type):
    if extract_type == "last_number":
        number = extract_number(text)
        if number is None:
            return ""
        return str(int(number)) if number == int(number) else str(number)
    if extract_type == "last_line":
        last = ""
        for line in (text or "").splitlines():
            if line.strip():
                last = line.strip()
        return last
    return (text or "").strip()          # "full": the whole thing

def check_answer(prediction, gold, check_type):
    if check_type == "numeric":
        p = extract_number(str(prediction))
        g = extract_number(str(gold))
        return p is not None and g is not None and abs(p - g) < 1e-6
    if check_type == "exact":
        return str(prediction).strip().casefold() == str(gold).strip().casefold()
    # "llm_judge": ask a cheap model whether the prediction matches the gold answer.
    prompt = (f"Reference answer: {gold}\nCandidate answer: {prediction}\n"
              "Is the candidate correct / equivalent to the reference? Answer yes or no.")
    reply, _ = _call_api("claude-haiku-4-5", prompt, 16)
    return reply.strip().lower().startswith("y")

**Examples — extract + check.** `extract_answer` is the *deterministic fallback* extractor (the profiler usually writes a task-specific one instead); `check_answer` is the grader. Pure functions, no API except `llm_judge`.

```python
extract_number("Sold 75 of 84 apples, so 9 remain.")   # -> 9.0
extract_number("no numbers here")                       # -> None

extract_answer("...therefore the answer is 9.", "last_number")  # -> "9"
extract_answer("reasoning\nfinal_label", "last_line")           # -> "final_label"
extract_answer("  keep it all  ", "full")                       # -> "keep it all"

check_answer("9", 9, "numeric")                # -> True
check_answer("9 apples left", "9", "numeric")  # -> True   (a number is pulled from each side)
check_answer("Positive", "positive", "exact")  # -> True   (case-insensitive)
check_answer("cat", "dog", "exact")            # -> False
# "llm_judge" also exists — it asks a cheap model to judge equivalence (so it hits the API).
```

## Step 1 · Profile the task — infer format, *write an extractor*, make data

One structured model call reads your `SEED_PROMPT` and returns: a `description`, the grading `CHECK_TYPE` (numeric / exact / llm_judge), and — instead of picking from a fixed menu — an **`extract(text)` function written for your task**, plus a few labeled *probes*. `build_extractor` compiles that function and checks it recovers the gold answer on every probe; if it compiles and passes we use it, otherwise we fall back to the deterministic `extract_answer` for the inferred `EXTRACT_TYPE`. This keeps extraction general (any answer format) while keeping the grader trustworthy (a bad extractor is caught, not silently used).

If you gave no `DATASET`, it also generates labeled examples (free-form reference answers when the check is an LLM judge, bare values otherwise). The data is split into **DEV** (which the design agent may test against) and a held-out **TEST** set used only for the final ranking. The cell prints what was inferred and whether the LLM-written extractor passed validation.

In [ ]:
# ---- Profile the task: infer format, WRITE an extractor, and get a dataset -
PROFILER_MODEL = "claude-opus-4-8"

def ask_for_json(model, prompt, schema):
    # One structured-output call: the reply is guaranteed to match `schema`.
    message = client.messages.create(
        model=model, max_tokens=4096, thinking={"type": "disabled"},
        output_config={"format": {"type": "json_schema", "schema": schema}},
        messages=[{"role": "user", "content": prompt}])
    text = ""
    for block in message.content:
        if block.type == "text":
            text += block.text
    return json.loads(text)

PROFILE_SCHEMA = {
    "type": "object",
    "properties": {
        "description":  {"type": "string"},
        "answer_type":  {"type": "string"},
        "check_type":   {"type": "string", "enum": ["numeric", "exact", "llm_judge"]},
        "extract_type": {"type": "string", "enum": ["last_number", "last_line", "full"]},
        "extract_code": {"type": "string"},
        "extract_probes": {"type": "array", "items": {
            "type": "object",
            "properties": {"text": {"type": "string"}, "expected": {"type": "string"}},
            "required": ["text", "expected"], "additionalProperties": False}},
    },
    "required": ["description", "answer_type", "check_type", "extract_type",
                 "extract_code", "extract_probes"],
    "additionalProperties": False,
}

def profile_task(seed_prompt, dataset):
    examples = ""
    if dataset:
        examples = "\n\nExamples (input -> answer):\n"
        for item in dataset[:5]:
            examples += f"- {item['question']} -> {item['answer']}\n"
    prompt = (
        f"A user wants to build an LLM workflow for this task:\n{seed_prompt}{examples}\n\n"
        "Reply with:\n"
        "- description: one clear paragraph describing the task for a workflow designer.\n"
        "- answer_type: a short label (e.g. 'integer', 'category', 'free text').\n"
        "- check_type: how to score a prediction against the gold answer — 'numeric' for "
        "numbers, 'exact' for short labels/strings, 'llm_judge' for free-form/semantic answers.\n"
        "- extract_code: Python source defining `def extract(text): ...` that pulls ONLY the "
        "final answer out of a model's raw response and returns it as a string, for THIS task's "
        "answer format. It may use `re`, `json`, and a helper `extract_number(text) -> float|None`; "
        "NO imports and no I/O. Handle verbose responses (reasoning, then the answer) and return "
        "'' when nothing is found. For a free-form task it can simply return the trimmed text.\n"
        "- extract_probes: 3-4 objects {\"text\", \"expected\"} where `text` is a REALISTIC, "
        "verbose model response for this task (reasoning, then the answer, with varied phrasings "
        "and answer positions) and `expected` is exactly what extract(text) should return. These "
        "validate extract_code, so make the texts non-trivial.\n"
        "- extract_type: a deterministic fallback among 'last_number' / 'last_line' / 'full', "
        "used only if extract_code fails validation.")
    return ask_for_json(PROFILER_MODEL, prompt, PROFILE_SCHEMA)

def compile_extractor(code):
    # Run the profiler's extractor source and return its extract() function.
    namespace = {"re": re, "json": json, "extract_number": extract_number}
    exec(code, namespace)
    return namespace["extract"]

def build_extractor(profile):
    # Prefer the LLM-written extractor, but only trust it if it round-trips the
    # profiler's gold probes; otherwise fall back to the deterministic extractor.
    fallback_type = profile["extract_type"]
    check_type = profile["check_type"]
    def deterministic(text):
        return extract_answer(text, fallback_type)

    code = (profile.get("extract_code") or "").strip()
    probes = profile.get("extract_probes") or []
    if not code:
        return deterministic, "no extract_code"
    if len(probes) < 2:
        return deterministic, "too few probes to validate"
    # At least one probe must be verbose, else the validation proves nothing
    # (a trivial "42" -> "42" probe would pass any extractor).
    verbose = False
    for probe in probes:
        if len(probe["text"].strip()) >= len(str(probe["expected"]).strip()) + 15:
            verbose = True
    if not verbose:
        return deterministic, "probes too trivial to trust"

    try:
        extract = compile_extractor(code)
    except Exception as error:
        return deterministic, f"did not compile: {error}"
    for probe in probes:
        try:
            got = extract(probe["text"])
        except Exception as error:
            return deterministic, f"errored on a probe: {error}"
        if not check_answer(got, probe["expected"], check_type):
            return deterministic, f"failed a probe (got {got!r}, expected {probe['expected']!r})"
    return extract, "ok"

DATASET_SCHEMA = {
    "type": "object",
    "properties": {"data": {"type": "array", "items": {
        "type": "object",
        "properties": {"question": {"type": "string"}, "answer": {"type": "string"}},
        "required": ["question", "answer"], "additionalProperties": False}}},
    "required": ["data"], "additionalProperties": False,
}

def generate_dataset(profile, n=12):
    # For LLM-judged (free-form) tasks the 'answer' is an ideal reference output;
    # otherwise it's a bare value (a number or a short label).
    if profile["check_type"] == "llm_judge":
        answer_rule = ("Each 'answer' is an ideal reference output for that input — it may be "
                       "multi-sentence / free-form; it will be graded by an LLM judge.")
    else:
        answer_rule = ("Each 'answer' must be the correct final target ONLY — a bare value "
                       "(the number or the label), with no explanation or units.")
    prompt = (f"Generate {n} diverse labeled examples for this task:\n{profile['description']}\n\n"
              + answer_rule + " Make them solvable and unambiguous.")
    return ask_for_json(PROFILER_MODEL, prompt, DATASET_SCHEMA)["data"]

profile = profile_task(SEED_PROMPT, DATASET)
EXTRACT_TYPE = profile["extract_type"]
CHECK_TYPE   = profile["check_type"]
EXTRACT, extract_status = build_extractor(profile)   # the resolved extractor (LLM-written or fallback)
EXTRACT_IS_LLM = (extract_status == "ok")

if DATASET is not None:
    DATA = DATASET
else:
    DATA = generate_dataset(profile)

# Split: the designer may self-test on DEV; the final ranking is on held-out TEST.
split = max(1, len(DATA) * 3 // 5)
DATA_DEV = DATA[:split]
DATA_TEST = DATA[split:]

print(f"answer_type = {profile['answer_type']}   check = {CHECK_TYPE}")
if EXTRACT_IS_LLM:
    print(f"extractor   = LLM-written, validated on {len(profile['extract_probes'])} probes")
else:
    print(f"extractor   = deterministic '{EXTRACT_TYPE}'  (fallback: {extract_status})")
print(f"{len(DATA)} examples  ->  {len(DATA_DEV)} dev / {len(DATA_TEST)} test")
print("Description:", profile["description"])

**Examples — profiling.** `profile_task` infers the format, *writes* an extractor, and returns probes to validate it; `build_extractor` compiles + validates it (or falls back); `generate_dataset` makes labeled examples.

```python
profile_task("Grade-school math; answers are integers.", None)
# -> {"description": "...", "answer_type": "integer", "check_type": "numeric",
#     "extract_code": "def extract(text):\n    n = extract_number(text)\n    return '' if n is None else str(int(n))",
#     "extract_probes": [{"text": "She sold 75, so 9 are left.", "expected": "9"}, ...],
#     "extract_type": "last_number"}                        # deterministic fallback

build_extractor(profile)   # -> (extract_fn, "ok")   or   (deterministic_fn, "failed a probe (...)")

generate_dataset(profile, n=3)
# -> [{"question": "A store had 84 apples...", "answer": "9"}, ...]
```

## Helpers for picking a workflow

Three small functions used in the final step, operating on a list of result dicts (`{"name", "accuracy", "cost_per_query"}`):
- `pareto_front(results)` — keep only the **non-dominated** results (see the examples below).
- `best_under_budget(results, max_cost)` — the most accurate result costing no more than `max_cost`.
- `cheapest_above_accuracy(results, min_accuracy)` — the cheapest result reaching `min_accuracy`.

In [ ]:
# ---- Pareto frontier + picking a workflow under a constraint --------------
def cost_of(result):
    return result["cost_per_query"]

def pareto_front(results):
    # Keep a result only if no OTHER result is at least as accurate AND at least
    # as cheap (and strictly better on at least one of the two).
    front = []
    for r in results:
        dominated = False
        for other in results:
            if other is r:
                continue
            at_least_as_cheap    = other["cost_per_query"] <= r["cost_per_query"]
            at_least_as_accurate = other["accuracy"] >= r["accuracy"]
            strictly_better      = (other["cost_per_query"] < r["cost_per_query"]
                                    or other["accuracy"] > r["accuracy"])
            if at_least_as_cheap and at_least_as_accurate and strictly_better:
                dominated = True
                break
        if not dominated:
            front.append(r)
    front.sort(key=cost_of)          # cheapest first
    return front

def best_under_budget(results, max_cost):
    best = None
    for r in results:
        if r["cost_per_query"] <= max_cost:
            if best is None or r["accuracy"] > best["accuracy"]:
                best = r
    return best

def cheapest_above_accuracy(results, min_accuracy):
    cheapest = None
    for r in results:
        if r["accuracy"] >= min_accuracy:
            if cheapest is None or r["cost_per_query"] < cheapest["cost_per_query"]:
                cheapest = r
    return cheapest

**Examples — Pareto selection** (pure functions, no API):

```python
results = [
    {"name": "cheap_direct",     "accuracy": 0.70, "cost_per_query": 0.0002},
    {"name": "cot",              "accuracy": 0.90, "cost_per_query": 0.0010},
    {"name": "self_consistency", "accuracy": 0.92, "cost_per_query": 0.0050},
    {"name": "dominated",        "accuracy": 0.80, "cost_per_query": 0.0020},
]

pareto_front(results)                   # -> cheap_direct, cot, self_consistency
# "dominated" drops out: "cot" is both cheaper AND more accurate than it.
best_under_budget(results, 0.0015)      # -> cot   (most accurate costing < $0.0015/query)
cheapest_above_accuracy(results, 0.90)  # -> cot   (cheapest that reaches 0.90 accuracy)
```

## Step 2 · The design agent (called once per optimization round)

`run_design_round(round_num, context)` runs the **Claude Agent SDK** agent in a subprocess and returns the workflows it proposed. On round 1 it designs a diverse initial set; on later rounds `context` shows it the best workflows so far so it can design **cheaper** ones that keep accuracy. `summarize_archive(archive)` builds that context — the current DEV Pareto frontier plus the most-accurate workflow's code, as a base to improve on.

The agent's method + its dev evaluator are the two `SKILL.md` skills under `skills/` (`workflow-design`, `workflow-eval`), copied into its `.claude/skills/` each round. It runs in a separate process (`run_proposer.py`) so its async loop stays isolated from the notebook kernel.

> Requires `claude-agent-sdk` + `ANTHROPIC_API_KEY`; each round makes real API calls (spends money).

In [ ]:
# ---- The design agent, wrapped so we can call it each round ----------------
import tempfile, pathlib, shutil, subprocess, sys

PROPOSER_MODEL = "claude-opus-4-8"

def summarize_archive(archive):
    # Context the agent sees each round: the current best tradeoffs (DEV accuracy
    # + cost), plus the code of the most accurate workflow as a base to improve on.
    if not archive:
        return ""
    dev_results = []
    for p in archive:
        dev_results.append({"name": p["name"], "accuracy": p["dev_accuracy"],
                            "cost_per_query": p["dev_cost"]})
    lines = []
    for r in pareto_front(dev_results):
        lines.append(f"- {r['name']}: accuracy {r['accuracy']:.2f}, ${r['cost_per_query']:.5f}/query")

    most_accurate = archive[0]
    for p in archive:
        if p["dev_accuracy"] > most_accurate["dev_accuracy"]:
            most_accurate = p
    lines.append(f"\nMost accurate so far is '{most_accurate['name']}' "
                 f"(accuracy {most_accurate['dev_accuracy']:.2f}, "
                 f"${most_accurate['dev_cost']:.5f}/query). Its code:\n{most_accurate['code']}")
    return "\n".join(lines)

def run_design_round(round_num, context):
    agent_dir = pathlib.Path(tempfile.mkdtemp(prefix=f"proposer_r{round_num}_"))
    # task_spec the dev evaluator (eval_candidate.py) reads: the RESOLVED extractor
    # (the LLM-written code if it passed validation, else None -> deterministic type)
    # plus the checker, so the agent grades candidates exactly as this notebook does.
    (agent_dir / "task_spec.json").write_text(json.dumps({
        "description": profile["description"],
        "extract": {"type": EXTRACT_TYPE,
                    "code": profile["extract_code"] if EXTRACT_IS_LLM else None},
        "check": {"type": CHECK_TYPE, "task": profile["description"]},
    }))
    (agent_dir / "dev_task.json").write_text(json.dumps(DATA_DEV[:5]))
    skills_dir = agent_dir / ".claude" / "skills"
    skills_dir.mkdir(parents=True)
    for skill_name in ("workflow-design", "workflow-eval"):
        shutil.copytree(pathlib.Path("skills") / skill_name, skills_dir / skill_name)

    facts = ("Available models, cheap -> expensive: " + ", ".join(MODELS) + ".\n"
             "Answers are scored by extract='" + EXTRACT_TYPE + "' then check='" + CHECK_TYPE + "'.")
    if round_num == 1:
        goal = ("Design 4-5 DIVERSE, working candidate workflows for this task, spanning "
                "cheap -> accurate:\n" + profile["description"])
    else:
        goal = ("Improve on the workflows found so far for this task:\n" + profile["description"]
                + "\n\nWorkflows so far:\n" + context
                + "\n\nDesign 3-4 NEW workflows that keep accuracy at least as high as the best "
                "one above but cost LESS per query — e.g. a cheaper model, fewer calls, routing "
                "easy inputs to the cheap model, or code execution instead of many samples.")
    prompt = (goal + "\n\n" + facts + "\n\n"
              "Use the workflow-design skill for the contract and the workflow-eval skill to test "
              "each candidate on the dev set. Write your final picks to programs.json and stop.")

    (agent_dir / "proposer_config.json").write_text(json.dumps({
        "model": PROPOSER_MODEL,
        "skills": ["workflow-design", "workflow-eval"],
        "allowed_tools": ["WebSearch", "WebFetch", "Bash", "Read", "Write", "Skill"],
        "prompt": prompt,
    }))

    runner = str(pathlib.Path("run_proposer.py").resolve())
    process = subprocess.Popen([sys.executable, runner], cwd=str(agent_dir),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    programs_file = agent_dir / "programs.json"
    if programs_file.exists():
        data = json.loads(programs_file.read_text())
        return data["programs"] if isinstance(data, dict) else data
    # salvage candidate .py files if programs.json is missing
    programs = []
    for f in sorted(agent_dir.glob("*.py")):
        if f.name in ("run_proposer.py", "eval_candidate.py"):
            continue
        code = f.read_text()
        if "def solve" in code:
            programs.append({"name": f.stem, "description": "(recovered)", "code": code})
    return programs

## The runtime that executes a candidate program

The machinery that actually *runs* a program:

- `Runtime` — a per-query object. Its `.llm(...)` method is the metered `llm` a program receives: it counts tokens, adds up cost, and raises if the program exceeds its budget (max calls / tokens). `_call_api` from earlier does the real API call underneath.
- `compile_solve(code)` — runs a program's source text and returns its `solve` function.
- `evaluate_program(program, dataset, model)` — runs the program on every example (a fresh `Runtime` each time), grades it with the profiler's `EXTRACT` + `check_answer`, and returns `{"name", "accuracy", "cost_per_query"}`.

In [ ]:
# ---- Run a workflow program, metering + capping every model call ----------
from collections import Counter

class Runtime:
    """A workflow program calls `runtime.llm(...)` for every model call. This is
    the one place cost is measured, and it stops the program if it goes over budget."""
    def __init__(self, default_model, max_calls=24, max_tokens=120_000):
        self.default_model = default_model
        self.max_calls = max_calls
        self.max_tokens = max_tokens
        self.calls = 0
        self.tokens = 0
        self.cost = 0.0

    def llm(self, prompt, max_tokens=256, model=None, system=None, tools=None, effort=None):
        if self.calls >= self.max_calls:
            raise RuntimeError("workflow exceeded its model-call budget")
        self.calls += 1
        if model not in PRICES:
            model = self.default_model
        text, usage = _call_api(
            model, str(prompt), int(max_tokens), system=system, tools=tools, effort=effort)
        self.tokens += usage["input"] + usage["output"] + usage["cache_write"] + usage["cache_read"]
        self.cost += cost_usd(model, usage)
        if self.tokens > self.max_tokens:
            raise RuntimeError("workflow exceeded its token budget")
        return text

def compile_solve(code):
    # Run the program's source and return its solve() function. The program may
    # use these names without importing them. (Model-written code runs with full
    # Python here — fine for a trusted research notebook; sandbox it for untrusted code.)
    namespace = {"re": re, "json": json, "statistics": statistics,
                 "Counter": Counter, "extract_number": extract_number, "MODELS": MODELS}
    exec(code, namespace)
    return namespace["solve"]

def evaluate_program(program, dataset, default_model):
    try:
        solve = compile_solve(program["code"])
    except Exception:
        return {"name": program["name"], "accuracy": 0.0, "cost_per_query": 0.0}

    n_correct = 0
    total_cost = 0.0
    for item in dataset:
        runtime = Runtime(default_model)
        try:
            raw = solve(item["question"], runtime.llm)
            prediction = EXTRACT(str(raw))          # the resolved extractor (LLM-written or fallback)
            if check_answer(prediction, item["answer"], CHECK_TYPE):
                n_correct += 1
        except Exception:
            pass                     # over budget, or a bug in the program -> counts as wrong
        total_cost += runtime.cost

    return {"name": program["name"],
            "accuracy": n_correct / len(dataset),
            "cost_per_query": total_cost / len(dataset)}

**Examples — running a workflow program.** A program is source code defining `solve(question, llm)`. `compile_solve` turns the source into that function; `evaluate_program` runs it over the data.

```python
code = '''
def solve(question, llm):
    return llm(question + "  Answer with just the number.")
'''
solve = compile_solve(code)             # -> a function(question, llm)

# Inside a program, `llm` can be called any of these ways:
llm("What is 2+2?")                              # cheapest default model
llm("a hard one", model="claude-opus-4-8")       # route to a bigger model
llm("compute 47*53", tools=["code_execution"])   # let the model run Python
llm("...", effort="high")                        # let the model think harder

evaluate_program(program, dataset, "claude-haiku-4-5")
# -> {"name": "cot", "accuracy": 0.8, "cost_per_query": 0.00072}
```

## Step 3 · Optimize — loop the designer to cut cost without losing accuracy

Run the designer `N_ROUNDS` times. Each round it tries to beat the current best on **cost** while holding **accuracy**; every new candidate is scored on the **DEV** split and added to the `archive`. Over the rounds the archive fills with progressively more efficient workflows.

> This is the most expensive cell — it runs the agent `N_ROUNDS` times and scores every candidate on DEV, so it makes a lot of API calls. Lower `N_ROUNDS` (e.g. to 1) to spend less.

In [ ]:
# ---- The optimization loop -------------------------------------------------
N_ROUNDS = 3
BASE_MODEL = MODELS[0]     # a workflow's default model; it may escalate itself
archive = []               # every candidate seen: {name, description, code, dev_accuracy, dev_cost}

for round_num in range(1, N_ROUNDS + 1):
    print(f"\n===== design round {round_num} / {N_ROUNDS} =====")
    context = summarize_archive(archive)                       # empty on round 1
    for program in run_design_round(round_num, context):
        if any(program["code"] == seen["code"] for seen in archive):   # skip exact repeats
            continue
        scored = evaluate_program(program, DATA_DEV, BASE_MODEL)        # score on DEV
        program["dev_accuracy"] = scored["accuracy"]
        program["dev_cost"] = scored["cost_per_query"]
        archive.append(program)
        print(f"  + {program['name']:22s} dev acc {program['dev_accuracy']:.2f}  "
              f"${program['dev_cost']:.5f}/query")

print(f"\n{len(archive)} workflows in the archive after {N_ROUNDS} rounds.")

## Rank the finalists on held-out TEST

The DEV scores guided the loop; now we get honest numbers on the **TEST** split (which nothing was tuned against). To keep cost down we only TEST-evaluate the DEV Pareto frontier — the candidates actually worth choosing between. This produces `code_results`.

In [ ]:
# ---- Rank the finalists on the held-out TEST split -------------------------
# DEV scores guided the loop; now we report honest numbers on TEST (which nothing
# was tuned against). To keep cost down we only TEST-evaluate the DEV Pareto
# frontier — the candidates actually worth choosing between.
dev_results = []
for p in archive:
    dev_results.append({"name": p["name"], "accuracy": p["dev_accuracy"],
                        "cost_per_query": p["dev_cost"],
                        "code": p["code"], "description": p.get("description", "")})

finalists = pareto_front(dev_results)     # the non-dominated workflows on DEV

code_results = []
for p in finalists:
    scored = evaluate_program(p, DATA_TEST, BASE_MODEL)
    scored["description"] = p["description"]
    scored["code"] = p["code"]
    code_results.append(scored)

pd.DataFrame([{"name": r["name"], "accuracy": r["accuracy"],
               "cost_per_query": r["cost_per_query"]} for r in code_results]).sort_values(
    ["accuracy", "cost_per_query"], ascending=[False, True]).reset_index(drop=True)

## The tradeoffs — frontier + two automatic picks

The **Pareto frontier** over the finalists (cheapest → most accurate), plus two convenience picks — the best workflow under a dollar budget, and the cheapest above an accuracy floor — and a plot of every finalist with the frontier drawn through them.

In [ ]:
# ---- The Pareto frontier, the two constrained picks, and a plot ------------
frontier = pareto_front(code_results)

print("Pareto frontier (cheapest -> most accurate):")
for r in frontier:
    print(f"  {r['name']:22s}  accuracy {r['accuracy']:.2f}   ${r['cost_per_query']:.5f}/query")

BUDGET = 0.002
best = best_under_budget(code_results, BUDGET)
if best is None:
    print(f"\nNo program costs under ${BUDGET}/query.")
else:
    print(f"\nBest program under ${BUDGET}/query:  {best['name']}"
          f"  (accuracy {best['accuracy']:.2f}, ${best['cost_per_query']:.5f})")

TARGET = 0.80
cheapest = cheapest_above_accuracy(code_results, TARGET)
if cheapest is None:
    print(f"No program reaches accuracy {TARGET}.")
else:
    print(f"Cheapest program with accuracy >= {TARGET}:  {cheapest['name']}"
          f"  (accuracy {cheapest['accuracy']:.2f}, ${cheapest['cost_per_query']:.5f})")

# Plot every program as a point; draw a line through the frontier.
frontier_costs = []
frontier_accuracies = []
for r in frontier:
    frontier_costs.append(r["cost_per_query"])
    frontier_accuracies.append(r["accuracy"])

plt.figure(figsize=(7, 5))
for r in code_results:
    plt.scatter(r["cost_per_query"], r["accuracy"], color="#888888")
    plt.annotate(r["name"], (r["cost_per_query"], r["accuracy"]),
                 fontsize=8, xytext=(5, 4), textcoords="offset points")
plt.plot(frontier_costs, frontier_accuracies, "-o", color="#d9534f", label="Pareto frontier")
plt.xlabel("cost per query (USD)")
plt.ylabel("accuracy")
plt.title("LLM-designed workflows: accuracy vs. cost")
plt.legend()
plt.tight_layout()
plt.show()

## Step 4 · Choose your methodology

Every workflow below is on the frontier — a genuine option at a different accuracy/cost point. Read each one's code, then set `CHOICE` to the name you want; `chosen["code"]` is what you'd ship.

In [ ]:
# ---- Choose your methodology ----------------------------------------------
# Each workflow on the frontier is a real option — a different accuracy/cost
# point. Read them, then set CHOICE to the one you want; `chosen["code"]` is the
# workflow you'd ship.
for r in pareto_front(code_results):
    print("=" * 72)
    print(f"{r['name']}   —   accuracy {r['accuracy']:.2f},  ${r['cost_per_query']:.5f}/query")
    if r.get("description"):
        print(r["description"])
    print("-" * 72)
    print(r["code"].strip())
    print()

CHOICE = pareto_front(code_results)[-1]["name"]   # default: the most accurate frontier workflow
chosen = None
for r in code_results:
    if r["name"] == CHOICE:
        chosen = r

print("=" * 72)
print(f"CHOSEN: {CHOICE}   (edit CHOICE above to pick a different tradeoff)\n")
print(chosen["code"].strip())